In [2]:
# 분기 chain : LangChain + Chroma + LLM
# 문서 분할 -> 임베딩 -> ChromaDB에 저장 -> 질문 기반 문서 검색 -> 프롬프트 완성

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import CharacterTextSplitter

import os
from dotenv import load_dotenv

def func(text):
    return text.upper()

obj = RunnableLambda(func)
print(obj.invoke("hello"))
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY 환경 변수가 설정되지 않았습니다.")

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.5)

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/home/tankcc/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HELLO


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7758.86it/s]


In [ ]:
docs = [
    Document(
        page_content="LangChain is a framework for developing applications powered by language models."
    ),
    Document(
        page_content="Chroma is an open-source embedding database that allows you to store and query embeddings."
    ),
]

# 문서 분할
text_splitter = CharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 20,
    separator = "\n\n"
)

split_docs = text_splitter.split_documents(docs)
print(f"문서 분할 결과: {len(split_docs)}개의 청크로 분할되었습니다.")
print("분할된 문서 내용:    ")


# ChromaDB에 저장
db = Chroma.from_documents(
    documents = split_docs,
    embedding = embedding_model
    )

##########################################################################################
# Retriver : 사용자의 질문과 유사한 문서를 ChromaDB에서 검색하는 역할
retriever = db.as_retriever()
##########################################################################################

문서 분할 결과: 2개의 청크로 분할되었습니다.
분할된 문서 내용:    


In [5]:
# Prompt Template
prompt_template = PromptTemplate(
    input_variables = ["context", "question"],
    template = """
    You are a helpful assistant that provides answers based on the provided context.

    문서 내용:
    {context}

    질문:
    {question}

    답변은 5행 정도의 분량으로 해줘
    """
)

# chain구성 : LCEL 사용

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


############################################################################################################
# 프롬프트 완성
# rag_chain : 추상화
rag_chain = (
    {
        # 질문 유사 문장 얻기 -> retriever로 검색 -> 문서를 문자열로 포맷 -> 프롬프트 완성 -> LLM으로 답변 생성
        "context":retriever | RunnableLambda(format_docs),          # 검색된 문서를 문자열로 포맷, retriever로 검색된 문서를 format_docs 함수로 포맷
        "question":RunnablePassthrough()                            # 질문은 변형 없이 그대로 question으로 mapping
    }
    | prompt_template
    | llm
    | StrOutputParser()
)
############################################################################################################





############################################################################################################
# 질문
query = "RAG가 뭐야?"
print(f"질문: {query}")
############################################################################################################


############################################################################################################
# 결과
result = rag_chain.invoke(query)
print(f"답변: {result}")
############################################################################################################

질문: RAG가 뭐야?


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}